# M5 Forecasting Accuracy: Prophet

ลองใช้ Prophet แทน LightGBM (`modeling.ipynb`) ด้วย validation แบบเดียวกัน แล้วเทียบ WRMSSE

**ทำไมใช้แบบ top-down**
- Prophet fit ทีละ series (~2.5 วินาที) ถ้าทำทั้ง 30,490 สินค้า × ร้าน จะใช้เวลาราว 20 ชั่วโมง
- Prophet ไม่ได้ออกแบบมาสำหรับ series ที่ขายได้ 0 เกินครึ่งหนึ่งของวัน (EDA 6.5)
- จึง fit ที่ระดับ **ร้าน × แผนก (70 series)** ซึ่งยอดขายต่อเนื่อง แล้วแบ่งลงสินค้าตามสัดส่วนยอดขาย 28 วันสุดท้าย

**สิ่งที่ใส่ให้ Prophet** (จาก EDA Section 7)
- Weekly + yearly seasonality (EDA 6.2)
- Events เป็น holidays รวม Christmas ที่ร้านปิด (EDA 6.3)
- SNAP ของรัฐที่ร้านตั้งอยู่เป็น regressor (EDA 6.4)
- `seasonality_mode="multiplicative"` เพราะผลของวันในสัปดาห์และ event เป็นสัดส่วนของระดับยอดขาย

## 1. Setup

In [1]:
import logging
import time

import numpy as np
import pandas as pd
import plotly.graph_objects as go
from prophet import Prophet

from wrmsse import HORIZON, WRMSSE

logging.getLogger("cmdstanpy").disabled = True

DATA = "m5-data"
TRAIN_END = 1913
FINAL_END = 1941
GROUP = ["store_id", "dept_id"]

sales_wide = pd.read_csv(f"{DATA}/sales_train_evaluation.csv")
calendar = pd.read_csv(f"{DATA}/calendar.csv", parse_dates=["date"])
prices = pd.read_csv(f"{DATA}/sell_prices.csv")
calendar["day"] = calendar["d"].str[2:].astype(int)
dates = calendar.set_index("day")["date"]

## 2. Store × Department Series
ยอดขายรวมรายวันของแต่ละร้าน × แผนก (10 ร้าน × 7 แผนก = 70 series)

In [2]:
day_cols = [f"d_{d}" for d in range(1, FINAL_END + 1)]
group_sales = sales_wide.groupby(GROUP)[day_cols].sum()
group_sales.columns = range(1, FINAL_END + 1)
print(group_sales.shape)

holidays = (
    calendar.loc[calendar["event_name_1"].notna(), ["event_name_1", "date"]]
    .rename(columns={"event_name_1": "holiday", "date": "ds"})
)
snap = calendar.set_index("date")[["snap_CA", "snap_TX", "snap_WI"]]

(70, 1941)


## 3. Fit and Forecast
- fit 1 โมเดลต่อ series ด้วยข้อมูลตั้งแต่ `d_1` ถึงวันสุดท้ายของ train
- ค่าพยากรณ์ที่ติดลบปรับเป็น 0

In [3]:
def forecast_groups(train_end):
    """fit Prophet ทุก series ถึง train_end แล้วพยากรณ์ 28 วันถัดไป -> DataFrame (70 × 28)"""
    history_dates = dates.loc[1:train_end].to_numpy()
    future_dates = dates.loc[train_end + 1:train_end + HORIZON].to_numpy()
    forecasts = {}
    start = time.time()
    for (store, dept), y in group_sales.loc[:, 1:train_end].iterrows():
        snap_col = f"snap_{store.split('_')[0]}"
        history = pd.DataFrame({"ds": history_dates, "y": y.to_numpy()})
        history["snap"] = snap.loc[history["ds"], snap_col].to_numpy()

        model = Prophet(holidays=holidays, seasonality_mode="multiplicative",
                        weekly_seasonality=True, yearly_seasonality=True, daily_seasonality=False)
        model.add_regressor("snap")
        model.fit(history)

        future = pd.DataFrame({"ds": future_dates})
        future["snap"] = snap.loc[future["ds"], snap_col].to_numpy()
        forecasts[(store, dept)] = model.predict(future)["yhat"].clip(lower=0).to_numpy()
    print(f"Fitted {len(forecasts)} series in {time.time() - start:.0f}s")
    return pd.DataFrame.from_dict(forecasts, orient="index", columns=range(train_end + 1, train_end + HORIZON + 1))


def disaggregate(group_forecast, train_end):
    """แบ่งค่าพยากรณ์ร้าน × แผนก ลงสินค้า ตามสัดส่วนยอดขาย 28 วันสุดท้ายของ train -> array (30,490 × 28)"""
    recent = sales_wide[[f"d_{d}" for d in range(train_end - HORIZON + 1, train_end + 1)]].sum(axis=1)
    group_total = recent.groupby([sales_wide[c] for c in GROUP]).transform("sum")
    share = (recent / group_total).fillna(0).to_numpy()
    group_rows = group_forecast.reindex(pd.MultiIndex.from_frame(sales_wide[GROUP])).to_numpy()
    return share[:, None] * group_rows

## 4. Development: train ≤ `d_1913`, forecast `d_1914` – `d_1941`

In [4]:
dev_groups = forecast_groups(TRAIN_END)
dev_pred = disaggregate(dev_groups, TRAIN_END)

evaluator = WRMSSE(sales_wide, prices, calendar, TRAIN_END)
valid_cols = [f"d_{d}" for d in range(TRAIN_END + 1, TRAIN_END + HORIZON + 1)]
valid_actual = sales_wide[valid_cols].to_numpy(np.float64)
train_tail = sales_wide[[f"d_{d}" for d in range(TRAIN_END - HORIZON + 1, TRAIN_END + 1)]].to_numpy(np.float64)

results = {
    "Seasonal naive": evaluator.score(np.tile(train_tail[:, -7:], HORIZON // 7), valid_actual),
    "Prophet (top-down)": evaluator.score(dev_pred, valid_actual),
}

Fitted 70 series in 34s


**เทียบกับ LightGBM**
คะแนน LightGBM คัดลอกจากผลของ `modeling.ipynb` (Section 6) เพราะ train ใหม่ใช้เวลาราว 30 นาที

In [5]:
LIGHTGBM = pd.Series({
    "1 total": 0.5311, "2 state": 0.5359, "3 store": 0.5905, "4 category": 0.5327,
    "5 department": 0.5579, "6 state × category": 0.5569, "7 state × department": 0.5844,
    "8 store × category": 0.6244, "9 store × department": 0.6690, "10 item": 0.8313,
    "11 item × state": 0.8366, "12 item × store": 0.8375,
})
results["LightGBM (modeling.ipynb)"] = (LIGHTGBM.mean(), LIGHTGBM)

table = pd.DataFrame({name: r[1] for name, r in results.items()})
overall = pd.DataFrame({name: [r[0]] for name, r in results.items()}, index=["WRMSSE (mean of 12)"])
pd.concat([overall, table]).style.format("{:.4f}")

,Seasonal naive,Prophet (top-down),LightGBM (modeling.ipynb)
WRMSSE (mean of 12),0.8697,0.7038,0.6407
1 total,0.7514,0.6067,0.5311
2 state,0.7544,0.6029,0.5359
3 store,0.7993,0.6402,0.5905
4 category,0.7620,0.6227,0.5327
5 department,0.8171,0.6659,0.5579
6 state × category,0.7830,0.6407,0.5569
7 state × department,0.8304,0.6841,0.5844
8 store × category,0.8393,0.7010,0.6244
9 store × department,0.8919,0.7498,0.6690


**Total sales: forecast vs actual**

In [6]:
history_days = list(range(TRAIN_END - 55, TRAIN_END + HORIZON + 1))
actual_total = sales_wide[[f"d_{d}" for d in history_days]].sum()
forecast_dates = dates.loc[TRAIN_END + 1:TRAIN_END + HORIZON]

fig = go.Figure([
    go.Scatter(x=dates.loc[history_days], y=actual_total.to_numpy(), name="Actual", line=dict(color="#52514e", width=2)),
    go.Scatter(x=forecast_dates, y=dev_pred.sum(axis=0), name="Prophet (top-down)", line=dict(color="#2a78d6", width=2)),
])
fig.add_vline(x=dates.loc[TRAIN_END + 1], line_dash="dot", line_color="gray")
fig.update_layout(title="Total daily units: Prophet forecast vs actual (d_1914 – d_1941)",
                  yaxis_title="Units sold per day", hovermode="x unified", plot_bgcolor="white")
fig.show()
print(f"Forecast total: {dev_pred.sum():,.0f}  Actual total: {valid_actual.sum():,.0f}  "
      f"Diff: {dev_pred.sum() / valid_actual.sum() - 1:+.1%}")

Forecast total: 1,144,387  Actual total: 1,231,764  Diff: -7.1%


## 5. Final: train ≤ `d_1941`, forecast `d_1942` – `d_1969`
บันทึกเป็น `submission_prophet.csv` (แถว `_validation` = ยอดขายจริง `d_1914` – `d_1941`)

In [7]:
final_pred = disaggregate(forecast_groups(FINAL_END), FINAL_END)

f_cols = [f"F{i}" for i in range(1, HORIZON + 1)]
evaluation = pd.DataFrame(final_pred, columns=f_cols).assign(id=sales_wide["id"].to_numpy())
validation = pd.DataFrame(valid_actual, columns=f_cols).assign(
    id=sales_wide["id"].str.replace("_evaluation", "_validation").to_numpy()
)
sample = pd.read_csv(f"{DATA}/sample_submission.csv", usecols=["id"])
submission = sample.merge(pd.concat([validation, evaluation]), on="id", how="left")
assert submission[f_cols].notna().all().all() and len(submission) == len(sample)

submission.to_csv("submission_prophet.csv", index=False)
print(f"Saved submission_prophet.csv: {submission.shape}")

Fitted 70 series in 35s


Saved submission_prophet.csv: (60980, 29)


## 6. Summary

| Model | WRMSSE (`d_1914` – `d_1941`) |
|---|---|
| Seasonal naive | 0.8697 |
| Prophet (top-down จากร้าน × แผนก) | 0.7038 |
| **LightGBM** (`modeling.ipynb`) | **0.6407** |

- Prophet ดีกว่า seasonal naive 19% แต่ด้อยกว่า LightGBM 10% และด้อยกว่าในทุกระดับทั้ง 12 ระดับ
- **ระดับรวมห่างจาก LightGBM มากที่สุด** เช่น ยอดรวม 0.61 เทียบกับ 0.53 และแผนก 0.67 เทียบกับ 0.56
- **ระดับสินค้าใกล้เคียงกัน** (0.84 เทียบกับ 0.84) ทั้งที่ Prophet แบ่งลงสินค้าด้วยสัดส่วนคงที่ ซึ่งเป็นวิธีที่ง่ายมาก
- ยอดรวมที่พยากรณ์ได้ต่ำกว่ายอดจริง 7.1% (LightGBM ต่ำกว่า 6.2%) โมเดลทั้งสองพยากรณ์ช่วงนี้ต่ำกว่าจริงเหมือนกัน
- เร็วกว่ามาก: fit 70 series ใช้เวลา ~35 วินาที ขณะที่ LightGBM ใช้ ~15 นาทีต่อโมเดล

**ข้อจำกัดและสิ่งที่ลองต่อได้**
- ยังไม่ได้ปรับ parameter ของ Prophet เลย (เช่น `changepoint_prior_scale`, `seasonality_prior_scale`)
- ลองระดับอื่นในการ fit เช่น ร้าน × หมวด (30 series) หรือสินค้า × รัฐ
- สัดส่วนลงสินค้ามาจาก 28 วันสุดท้ายเท่านั้น อาจลองช่วงยาวกว่านี้ หรือใช้สัดส่วนตามวันในสัปดาห์
- ลองเอาค่าพยากรณ์ของ Prophet กับ LightGBM มาเฉลี่ยกัน (ensemble) เพราะสองโมเดลทำงานต่างกัน
